# Importación de Librerias

In [104]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

### Lectura de Archivo


In [105]:
encuesta=pd.read_csv('C:/Users/tomasito/Downloads/encuesta-anual-hogares-2019.csv', sep=',', encoding='latin1')

### Inspección 

In [106]:
#ver los datos que tiene el df
encuesta.head(5)

,id,nhogar,miembro,comuna,dominio,edad,sexo,parentesco_jefe,situacion_conyugal,num_miembro_padre,...,ingreso_per_capita_familiar,estado_educativo,sector_educativo,nivel_actual,nivel_max_educativo,años_escolaridad,lugar_nacimiento,afiliacion_salud,hijos_nacidos_vivos,cantidad_hijos_nac_vivos
0,1,1,1,5,Resto de la Ciudad,18,Mujer,Jefe,Soltero/a,Padre no vive en el hogar,...,9000,Asiste,Estatal/publico,Universitario,Otras escuelas especiales,12,PBA excepto GBA,Solo obra social,No,No corresponde
1,1,1,2,5,Resto de la Ciudad,18,Mujer,Otro no familiar,Soltero/a,Padre no vive en el hogar,...,9000,Asiste,Estatal/publico,Universitario,Otras escuelas especiales,12,Otra provincia,Solo plan de medicina prepaga por contratación...,No,No corresponde
2,2,1,1,2,Resto de la Ciudad,18,Varon,Jefe,Soltero/a,Padre no vive en el hogar,...,33333,Asiste,Privado religioso,Universitario,Otras escuelas especiales,12,CABA,Solo plan de medicina prepaga por contratación...,NaN,No corresponde
3,2,1,2,2,Resto de la Ciudad,50,Mujer,Padre/Madre/Suegro/a,Viudo/a,No corresponde,...,33333,No asiste pero asistió,No corresponde,No corresponde,Secundario/medio comun,17,CABA,Solo prepaga o mutual via OS,Si,2
4,2,1,3,2,Resto de la Ciudad,17,Varon,Otro familiar,Soltero/a,Padre no vive en el hogar,...,33333,Asiste,Privado religioso,Secundario/medio comun,EGB (1° a 9° año),10,CABA,Solo plan de medicina prepaga por contratación...,NaN,No corresponde


In [107]:
# Ver que cantidad de datas y tipos de datos 
encuesta.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14319 entries, 0 to 14318
Data columns (total 31 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   id                           14319 non-null  int64 
 1   nhogar                       14319 non-null  int64 
 2   miembro                      14319 non-null  int64 
 3   comuna                       14319 non-null  int64 
 4   dominio                      14319 non-null  object
 5   edad                         14319 non-null  int64 
 6   sexo                         14319 non-null  object
 7   parentesco_jefe              14319 non-null  object
 8   situacion_conyugal           14318 non-null  object
 9   num_miembro_padre            14319 non-null  object
 10  num_miembro_madre            14319 non-null  object
 11  estado_ocupacional           14319 non-null  object
 12  cat_ocupacional              14319 non-null  object
 13  calidad_ingresos_lab         14

In [108]:
#muestra solo las columnas que tienen datos nulos
encuesta.columns[encuesta.isnull().any()]


Index(['situacion_conyugal', 'sector_educativo', 'nivel_max_educativo',
       'años_escolaridad', 'lugar_nacimiento', 'afiliacion_salud',
       'hijos_nacidos_vivos'],
      dtype='object')

In [109]:
#muestra el numero de datos nulos por columna que contineen datos nulos
encuesta.isnull().sum()[encuesta.isnull().sum() > 0]


situacion_conyugal        1
sector_educativo          3
nivel_max_educativo    1054
años_escolaridad         62
lugar_nacimiento          1
afiliacion_salud          4
hijos_nacidos_vivos    7784
dtype: int64

### 1) Primera premisa 

In [110]:
# Eliminacion de la columna id e hijos nacidos vivios
encuesta.drop(columns=['id', 'hijos_nacidos_vivos'], inplace=True)

### Discretización 

### Ingresos Familiares

In [111]:
#Verificacion de datos nulos
encuesta['ingresos_familiares'].isnull().sum()

np.int64(0)

In [112]:
# Cuenta los valores
encuesta['ingresos_familiares'].value_counts()

ingresos_familiares
60000    280
40000    280
50000    265
30000    264
80000    242
        ... 
73537      1
10670      1
16140      1
11300      1
32300      1
Name: count, Length: 960, dtype: int64

In [113]:
#Discretizacion en 8 catogorias por igual frecuencia
ingresosf_cat, ingresosf_cat_bins = pd.qcut(encuesta['ingresos_familiares'],q=8,retbins=True,duplicates='drop')

In [114]:
#contamos los valores del objeto y ordenamos por el valor de los bins 
ingresosf_cat.value_counts().sort_index()

ingresos_familiares
(-0.001, 20800.0]        1796
(20800.0, 30000.0]       1810
(30000.0, 42000.0]       1841
(42000.0, 54000.0]       1721
(54000.0, 70000.0]       1905
(70000.0, 90000.0]       1781
(90000.0, 124000.0]      1677
(124000.0, 1000000.0]    1788
Name: count, dtype: int64

In [115]:
# Discretizacion en 9 categorias por rangos iguales
ingresosf_cat1, ingresosf_cat_bins1=pd.cut(encuesta['ingresos_familiares'], bins=8, retbins=True, duplicates='drop')

In [116]:
ingresosf_cat1.value_counts().sort_index()

ingresos_familiares
(-1000.0, 125000.0]      12573
(125000.0, 250000.0]      1482
(250000.0, 375000.0]       185
(375000.0, 500000.0]        41
(500000.0, 625000.0]        28
(625000.0, 750000.0]         9
(750000.0, 875000.0]         0
(875000.0, 1000000.0]        1
Name: count, dtype: int64

### conlcusión
 Se Observa una clara diferencia en los tipos de rangos asumido entre el pcut y el cuts y en la distribucion de la data 

### Ingreso percapita familiar

In [117]:
#Verificacion de datos nulos
encuesta.columns

Index(['nhogar', 'miembro', 'comuna', 'dominio', 'edad', 'sexo',
       'parentesco_jefe', 'situacion_conyugal', 'num_miembro_padre',
       'num_miembro_madre', 'estado_ocupacional', 'cat_ocupacional',
       'calidad_ingresos_lab', 'ingreso_total_lab', 'calidad_ingresos_no_lab',
       'ingreso_total_no_lab', 'calidad_ingresos_totales', 'ingresos_totales',
       'calidad_ingresos_familiares', 'ingresos_familiares',
       'ingreso_per_capita_familiar', 'estado_educativo', 'sector_educativo',
       'nivel_actual', 'nivel_max_educativo', 'años_escolaridad',
       'lugar_nacimiento', 'afiliacion_salud', 'cantidad_hijos_nac_vivos'],
      dtype='object')

In [118]:
encuesta['ingreso_per_capita_familiar'].isnull().sum()

np.int64(0)

In [119]:
encuesta['ingreso_per_capita_familiar'].value_counts()

ingreso_per_capita_familiar
20000     301
30000     255
40000     198
15000     196
12000     189
         ... 
143000      1
103000      1
31400       1
212500      1
24600       1
Name: count, Length: 1257, dtype: int64

In [120]:
ingresopercapita_cat,ingresopercapita_bins=pd.qcut(encuesta['ingreso_per_capita_familiar'],q=8,retbins=True,duplicates='drop')

In [121]:
ingresopercapita_cat.value_counts().sort_index()

ingreso_per_capita_familiar
(-0.001, 6250.0]        1807
(6250.0, 10500.0]       1783
(10500.0, 14350.0]      1784
(14350.0, 19900.0]      1786
(19900.0, 25000.0]      1790
(25000.0, 33500.0]      1803
(33500.0, 47925.0]      1776
(47925.0, 1000000.0]    1790
Name: count, dtype: int64

In [122]:
ingresospercapita_cat1,ingresopercapita_bins1=pd.cut(encuesta['ingreso_per_capita_familiar'],bins=8,retbins=True,duplicates='drop')

In [123]:
ingresospercapita_cat1.value_counts().sort_index()

ingreso_per_capita_familiar
(-1000.0, 125000.0]      14194
(125000.0, 250000.0]       110
(250000.0, 375000.0]        10
(375000.0, 500000.0]         4
(500000.0, 625000.0]         0
(625000.0, 750000.0]         0
(750000.0, 875000.0]         0
(875000.0, 1000000.0]        1
Name: count, dtype: int64

# Ingresos total lab

In [124]:
encuesta['ingreso_total_lab'].isnull().sum()

np.int64(0)

In [125]:
encuesta['ingreso_total_lab'].value_counts()

ingreso_total_lab
0         6963
30000      504
20000      396
40000      378
50000      323
          ... 
152000       1
41600        1
250          1
40800        1
20500        1
Name: count, Length: 440, dtype: int64

In [126]:
ingresos_total_lab_cat,ingreso_total_lab_bins=pd.qcut(encuesta['ingreso_total_lab'],q=10,retbins=True,duplicates='drop')

In [127]:
ingresos_total_lab_cat.value_counts().sort_index()

ingreso_total_lab
(-0.001, 2500.0]        7168
(2500.0, 15000.0]       1499
(15000.0, 25000.0]      1397
(25000.0, 37000.0]      1431
(37000.0, 56000.0]      1397
(56000.0, 1000000.0]    1427
Name: count, dtype: int64

# Ingreso total no lab

In [128]:
encuesta['ingreso_total_no_lab'].isnull().sum()

np.int64(0)

In [129]:
encuesta['ingreso_total_no_lab'].value_counts()

ingreso_total_no_lab
0         10247
12000       374
13000       158
20000       154
12500       126
          ...  
22300         1
17600         1
227500        1
10015         1
12250         1
Name: count, Length: 375, dtype: int64

In [130]:
ingreso_total_no_lab_cat,ingreso_total_no_lab_bins=pd.qcut(encuesta['ingreso_total_no_lab'],q=4,retbins=True, duplicates='drop')

In [131]:
ingreso_total_no_lab_cat.value_counts().sort_index()

ingreso_total_no_lab
(-0.001, 4000.0]      10741
(4000.0, 500000.0]     3578
Name: count, dtype: int64

# Edad

In [132]:
encuesta['edad'].isnull().sum()

np.int64(0)

In [133]:
edad_cat,edad_bins=pd.cut(encuesta['edad'],bins=5,retbins=True,duplicates='drop')

In [134]:
edad_cat.value_counts().sort_index()

edad
(-0.1, 20.0]     3669
(20.0, 40.0]     4204
(40.0, 60.0]     3452
(60.0, 80.0]     2448
(80.0, 100.0]     546
Name: count, dtype: int64

# Preparación 

# Cambio de datos communa y nhogar

Se realiza el cambio porque dichas columnas a pesar de que son numerica son una connotacion 

In [135]:
encuesta['comuna'].dtypes

dtype('int64')

In [136]:
encuesta['comuna']=encuesta['comuna'].astype('str')

In [137]:
encuesta['nhogar'].dtypes

dtype('int64')

In [138]:
encuesta['nhogar']=encuesta['nhogar'].astype('str')

## Cambio de dato años de escolaridad 
Cambio de dato debido a como se llena la encuesta el cual esta basado en numero pero en string 

In [139]:
#viendo el tipo de dato
encuesta['años_escolaridad'].dtype

dtype('O')

In [140]:
#que cuente y muestr los valores
encuesta['años_escolaridad'].value_counts()

años_escolaridad
12                                    2824
17                                    1851
15                                    1498
7                                     1263
Ningun año de escolaridad aprobado    1226
14                                     722
9                                      588
19                                     557
10                                     539
13                                     483
11                                     451
8                                      397
16                                     334
3                                      261
2                                      241
6                                      229
5                                      224
4                                      211
1                                      196
18                                     162
Name: count, dtype: int64

In [141]:
encuesta['años_escolaridad'].unique()


array(['12', '17', '10', '8', 'Ningun año de escolaridad aprobado', '11',
       '9', '13', '7', '16', '14', '15', '5', '6', '2', '19', '4', '1',
       '3', '18', nan], dtype=object)

In [142]:
#modidicar valor de una columna
encuesta['años_escolaridad']=encuesta['años_escolaridad'].replace({'Ningun año de escolaridad aprobado':'0'})

In [143]:
#verificacion 
encuesta['años_escolaridad'].value_counts()

años_escolaridad
12    2824
17    1851
15    1498
7     1263
0     1226
14     722
9      588
19     557
10     539
13     483
11     451
8      397
16     334
3      261
2      241
6      229
5      224
4      211
1      196
18     162
Name: count, dtype: int64

# cambiamos la columna anterior a entero

In [144]:
#cambiamos el tipo de dato pero como la columna tien valores nulos se pasa primero a float y luego a Int32
encuesta['años_escolaridad']=encuesta['años_escolaridad'].astype('float').astype('Int32')
#verificacion de la columna
encuesta['años_escolaridad'].dtype

Int32Dtype()

In [145]:
#discretizacion en 5 categorias por igual frecuencia
encuesta_cat, encuesta_bins=pd.qcut(encuesta['años_escolaridad'],q=5,retbins=True,duplicates='drop')

In [148]:
#porcentaje de datos nulos en esa columna
encuesta['años_escolaridad'].isnull().sum() / len(encuesta['años_escolaridad']) * 100


np.float64(0.43299113066554923)

In [149]:
#borrar los datos nulos de la columna años escolariad y verficar que se borraron
encuesta = encuesta.dropna(subset=['años_escolaridad'])
encuesta['años_escolaridad'].isnull().sum()

np.int64(0)

In [150]:
encuesta['situacion_conyugal'].isnull().sum() / len(encuesta['situacion_conyugal']) * 100

np.float64(0.007014098337658693)

In [151]:
#verificacion de datos % de datos nulos
columnas=['situacion_conyugal','sector_educativo','lugar_nacimiento','afiliacion_salud']
resultados=[]
for columna in columnas:
    resultados.append(encuesta[columna].isnull().sum() / len(encuesta[columna]) * 100)
    print(f'Porcentaje de datos nulos en {columna}: {resultados[-1]:.2f}%')
    
    


Porcentaje de datos nulos en situacion_conyugal: 0.01%
Porcentaje de datos nulos en sector_educativo: 0.02%
Porcentaje de datos nulos en lugar_nacimiento: 0.01%
Porcentaje de datos nulos en afiliacion_salud: 0.03%


In [152]:
#borrar los datos nulos de las columas anteriores
encuesta=encuesta.dropna(subset=columnas)


In [153]:
#reset el indice
encuesta=encuesta.reset_index(drop=True)

In [ ]:
encuesta['desconocio']=encuesta

nivel_max_educativo
Secundario/medio comun       3670
Otras escuelas especiales    2565
EGB (1° a 9° año)            2293
Primario especial            2192
Sala de 5                    1534
Primario comun                942
Name: count, dtype: int64